# 과제 3 — ACT chunk_size 비교 실험

**과제 내용**: 같은 데이터셋으로 `chunk_size`만 10, 50, 100으로 바꾼 ACT 정책 3개를 학습하고, 실제 로봇에서 각각 10회씩 평가해 chunk_size가 성공률·동작의 부드러움·반응성에 주는 영향을 분석한다. chunk_size=100 정책에는 temporal ensemble을 적용해 10회 더 평가하고 결과를 비교한다. 학습된 정책 3개는 Huggingface Hub에 업로드한다.

### 실습 환경 — **LeRobot 0.6.0**
- Python 3.12 이상, PyTorch 2.7 이상이 필요하다.
- 실습 PC에는 `lerobot[all]==0.6.0`이 설치되어 있다. 개인 PC에서 하려면 같은 버전으로 설치한다.
  ```bash
  pip install "lerobot[all]==0.6.0"
  ```
  `all`에는 학습(wandb), 녹화·텔레오퍼레이션, SO-101 모터(feetech), Diffusion Policy, SmolVLA에 필요한 패키지가 모두 들어 있다. (0.6.0부터 extra 없이 `pip install lerobot`만 하면 학습·데이터셋 의존성이 빠지므로 주의)
- `lerobot-rollout`으로 녹화하는 평가 데이터셋은 이름이 **`rollout_`로 시작**해야 한다. `lerobot-record`와 `lerobot-rollout` 모두 새 데이터셋 이름 끝에 녹화 시각(`_YYYYMMDD_HHMMSS`)을 자동으로 붙이므로, 0절에는 **Hub에 실제로 올라간 이름**을 적는다.

### 작성 요령
- 이 노트북을 본인 GitHub 저장소(과제 1, 2와 같은 저장소, assignments 폴더)에 `assignment_3.ipynb`로 올린다.
- 학습과 평가는 **터미널에서** 진행하고 이 노트북에는 결과 확인 코드와 분석을 작성한다. (보고서를 노트북으로 작성하는 것이다.)
- 모든 코드 셀은 **실행한 뒤 출력 결과가 보이는 상태로** 커밋한다. (출력이 없으면 미제출로 간주)
- `[작성]` 표시가 있는 부분을 모두 채운다.

### 주의 사항
- 이번 과제는 **통제 실험**이다. 실험할 ACT 정책 3개는 `chunk_size`와 `n_action_steps`를 제외한 모든 설정(데이터셋, steps, batch_size, seed)이 같아야 한다. 1-3의 확인 셀에서 자동으로 검사한다.
- 과제 2의 정책(기본값 chunk_size=100)을 chunk_size=100 조건으로 **재사용해도 된다.** 이 경우 나머지 두 정책은 과제 2와 같은 데이터셋·steps·batch_size·seed로 학습해야 한다.
- 학습 데이터셋, 평가 데이터셋, 모델 repo는 모두 **public**으로 설정한다.
- 모델 가중치는 **Huggingface Hub에만** 올리고 GitHub에는 올리지 않는다. `.gitignore`에 `outputs/`가 있는지 확인할 것.
- **마감 시각 이후의 커밋은 반영하지 않는다.**

## 0. 제출 정보 및 개정 범위
2026-11-15 개정. 업로드된 `assignment_3(9).ipynb`의 전체 27개 셀 구조를 확인하여 본문을 갱신했다.
초기 실험은 부록 A와 `archive_hw3/assignment_3_initial_results.ipynb`에 보존했다. GIF·CSV·Hub 스냅샷도 함께 포함했다.
첨부파일 자체는 미작성 양식이므로 초기 결과는 이전 작업의 기존 assignments 노트북에서 복원했다.
**재학습 후 개선 실험의 정식 평가는 미평가**이다. 본문 실행은 보고서·설정 확인이며 로봇 학습/평가 실행을 뜻하지 않는다.


In [1]:
import json
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
STUDENT_ID, NAME, HF_USER = "2020151003", "고준협", "go234"
TASK = "Pick up the cube and put it in the box."
TRAIN_DATASET = "go234/cube-box_20261013_041703"
CHUNK_SIZES = [10, 50, 100]
POLICY_REPOS = {10:"go234/act_chunk10_b1_amp", 50:"go234/act_chunk50_b1_amp", 100:"go234/act_pick_cube"}
EVAL_DATASETS = {k:None for k in CHUNK_SIZES}  # 정식 녹화 후 실제 타임스탬프 포함 ID 입력
TE_EVAL_DATASET = None
ALLOW_NETWORK = False  # 기본은 저장한 공개 설정 스냅샷 사용
try: LEROBOT_VERSION = version("lerobot")
except PackageNotFoundError: LEROBOT_VERSION = None
print(STUDENT_ID, NAME)
print("보고서 커널 LeRobot:", LEROBOT_VERSION or "미설치 — 로봇 평가 환경과 별개")
print("새 정책:", POLICY_REPOS)
print("Chunk100: 과제 2 성공 모델 재사용. 과제 2 성공률을 새 평가로 전용하지 않음.")


2020151003 고준협
보고서 커널 LeRobot: 미설치 — 로봇 평가 환경과 별개
새 정책: {10: 'go234/act_chunk10_b1_amp', 50: 'go234/act_chunk50_b1_amp', 100: 'go234/act_pick_cube'}
Chunk100: 과제 2 성공 모델 재사용. 과제 2 성공률을 새 평가로 전용하지 않음.


In [2]:
snapshot_path = Path("evidence_hw3/retrained_hub_snapshot.json")
SNAPSHOT = json.loads(snapshot_path.read_text()) if snapshot_path.exists() else {"repos":{}}
def load_json(repo_id, filename, repo_type="model"):
    if not repo_id:
        return None
    if not ALLOW_NETWORK:
        value = SNAPSHOT.get("repos",{}).get(repo_id,{}).get(filename)
        if not isinstance(value,dict) or "_error" in value:
            print("미확인/대기:", repo_id, filename)
            return None
        return value
    try:
        import urllib.request
        prefix = "datasets/" if repo_type == "dataset" else ""
        with urllib.request.urlopen(f"https://huggingface.co/{prefix}{repo_id}/resolve/main/{filename}",timeout=15) as r:
            return json.load(r)
    except Exception as e:
        print("Hub 조회 미확인:", type(e).__name__, repo_id, filename)
        return None
print("공개 Hub 설정 조회 시각 UTC:", SNAPSHOT.get("checked_at", "미확인"))
print("저장된 설정 확인과 모델 가중치·로봇 성능 검증은 별개이다.")


공개 Hub 설정 조회 시각 UTC: 2026-11-15T13:01:58.277061+00:00
저장된 설정 확인과 모델 가중치·로봇 성능 검증은 별개이다.


## 1. 실험 설계

### 1-1. 배경
`chunk_size`(k)는 ACT가 한 번의 추론으로 예측하는 미래 행동의 개수다. 이번 실험에서는 `n_action_steps = chunk_size`로 두어 **예측한 청크를 끝까지 실행한 뒤 다시 추론**한다. (temporal ensemble은 5절에서 따로 진행하기 때문에 이 실험에는 사용하지 않는다.)

청크의 초 단위 길이는 데이터셋의 fps에 따라 달라진다.

$$T_{\text{chunk}} = \frac{\text{chunk\_size}}{\text{fps}}$$

- chunk_size가 **크면**: 예측 경로가 길게 유지될 수 있지만, 청크를 실행하는 동안 새 관측을 반영하지 못해 상황 변화에 둔감해진다.
- chunk_size가 **작으면**: 자주 새로 관측해 반응성은 좋지만, 단일 스텝 정책에 가까워져 오차 누적과 청크 경계의 덜컥거림이 늘 수 있다.

In [3]:
initial_snapshot_path = Path("evidence_hw3/hub_snapshot.json")
initial_snapshot = json.loads(initial_snapshot_path.read_text()) if initial_snapshot_path.exists() else {"repos":{}}
train_info = initial_snapshot.get("repos",{}).get(TRAIN_DATASET,{}).get("json",{}).get("meta/info.json",{})
fps = train_info.get("fps")
print("초기 스냅샷의 학습 fps:", fps, "(평가 fps=25와 구분)")
if fps:
    for k in CHUNK_SIZES: print(f"chunk{k}: 학습 시간 기준 {k/fps:.2f}s; 25Hz 평가 기준 {k/25:.2f}s")


초기 스냅샷의 학습 fps: 30 (평가 fps=25와 구분)
chunk10: 학습 시간 기준 0.33s; 25Hz 평가 기준 0.40s
chunk50: 학습 시간 기준 1.67s; 25Hz 평가 기준 2.00s
chunk100: 학습 시간 기준 3.33s; 25Hz 평가 기준 4.00s


### 1-2. 가설 및 재학습 이유
사전 작성 가설 기록은 없다. 짧은 청크는 더 잦은 관측 갱신, 긴 청크는 경로 연속성에 유리할 수 있다는 이론적 예상만 둔다.
과제 2 성공 모델과 초기 Chunk100은 ACT 구조·데이터셋이 같았지만 **batch_size 1 대 8, AMP true 대 false**였다.
따라서 과제 2 모델과 초기 모델을 묶어 chunk_size 효과를 비교하면 통제 요건을 충족하지 않는다.
초기 세 모델끼리의 비교와 별개로, 성공 기준 모델과 학습 조건을 맞추기 위해 Chunk10·50을 batch1+AMP로 재학습하고 Chunk100은 과제 2 모델을 재사용했다.
batch와 AMP를 동시에 변경했으므로 개선 원인을 둘 중 하나로 분리할 수 없으며, 새 정식 평가 없이 효과 크기를 확정하지 않는다.

### 1-3. 통제 변수: 재학습 후 개선 실험
|조건|Chunk10|Chunk50|Chunk100|
|---|---|---|---|
|정책|go234/act_chunk10_b1_amp|go234/act_chunk50_b1_amp|go234/act_pick_cube (재사용)|
|chunk_size / n_action_steps|10 / 10|50 / 50|100 / 100|
|dataset|go234/cube-box_20261013_041703|동일|동일|
|steps|100000|100000|100000|
|batch_size|1|1|1|
|seed|1000|1000|1000|
|AMP|true|true|true|
아래 코드는 공통 필드가 모두 누락된 경우를 일치로 처리하지 않고, 기대값과 비교한다. 나머지 정책 구조도 별도로 비교한다.


In [4]:
expected = dict(dataset=TRAIN_DATASET, steps=100000, batch_size=1, seed=1000, use_amp=True)
configs, control_checks = {}, {}
for k in CHUNK_SIZES:
    pc = load_json(POLICY_REPOS[k],"config.json") or {}
    tc = load_json(POLICY_REPOS[k],"train_config.json") or {}
    actual = dict(dataset=tc.get("dataset",{}).get("repo_id"), steps=tc.get("steps"), batch_size=tc.get("batch_size"), seed=tc.get("seed"), use_amp=pc.get("use_amp"))
    configs[k] = pc
    for field,value in expected.items():
        control_checks[f"chunk{k}/{field}"] = actual.get(field) == value and actual.get(field) is not None
    control_checks[f"chunk{k}/chunk_size"] = pc.get("chunk_size") == k
    control_checks[f"chunk{k}/n_action_steps"] = pc.get("n_action_steps") == k
    control_checks[f"chunk{k}/TE 비활성"] = "temporal_ensemble_coeff" in pc and pc["temporal_ensemble_coeff"] is None
    print(k, POLICY_REPOS[k], actual)
# 저장 위치·장치·정책 식별자는 구조 비교에서 제외
excluded = {"chunk_size","n_action_steps","repo_id","device","push_to_hub","pretrained_path"}
keys = set().union(*(p.keys() for p in configs.values())) - excluded
structure_diffs = [f for f in sorted(keys) if any(f not in configs[k] for k in CHUNK_SIZES) or len({json.dumps(configs[k].get(f),sort_keys=True) for k in CHUNK_SIZES}) != 1]
print("공통 조건 및 청크 기대값:", "통과" if all(control_checks.values()) else "불일치/미확인")
print("그 외 정책 설정 차이:", structure_diffs or "없음")


10 go234/act_chunk10_b1_amp {'dataset': 'go234/cube-box_20261013_041703', 'steps': 100000, 'batch_size': 1, 'seed': 1000, 'use_amp': True}
50 go234/act_chunk50_b1_amp {'dataset': 'go234/cube-box_20261013_041703', 'steps': 100000, 'batch_size': 1, 'seed': 1000, 'use_amp': True}
100 go234/act_pick_cube {'dataset': 'go234/cube-box_20261013_041703', 'steps': 100000, 'batch_size': 1, 'seed': 1000, 'use_amp': True}
공통 조건 및 청크 기대값: 통과
그 외 정책 설정 차이: 없음


## 2. 학습
### 2-1. 재학습 조건을 반영한 명령어
사용자가 보고한 재학습 조건이며 공개 Hub 설정과 대조했다. 원본 터미널 로그가 없으므로 아래 텍스트가 그대로 실행된 이력이라고 주장하지 않는다. 이 작업에서는 학습을 다시 실행하지 않았다.
```bash
lerobot-train \
  --dataset.repo_id=go234/cube-box_20261013_041703 \
  --policy.type=act \
  --policy.chunk_size=10 \
  --policy.n_action_steps=10 \
  --policy.device=cuda \
  --policy.use_amp=true \
  --batch_size=1 \
  --steps=100000 \
  --seed=1000 \
  --output_dir=outputs/train/act_chunk10_b1_amp \
  --job_name=act_chunk10_b1_amp \
  --policy.repo_id=go234/act_chunk10_b1_amp

lerobot-train \
  --dataset.repo_id=go234/cube-box_20261013_041703 \
  --policy.type=act \
  --policy.chunk_size=50 \
  --policy.n_action_steps=50 \
  --policy.device=cuda \
  --policy.use_amp=true \
  --batch_size=1 \
  --steps=100000 \
  --seed=1000 \
  --output_dir=outputs/train/act_chunk50_b1_amp \
  --job_name=act_chunk50_b1_amp \
  --policy.repo_id=go234/act_chunk50_b1_amp
# Chunk100: 과제 2 정책 go234/act_pick_cube 재사용 (재학습 없음)
```
### 2-2. 환경
공통 100000 steps, seed=1000, batch_size=1, AMP=true. GPU 모델·소요 시간·학습 당시 라이브러리 버전은 미확인이다.
초기 모델 `go234/act_chunk10`, `act_chunk50`, `act_chunk100`은 batch_size=8, AMP=false인 별도 실험으로 보존한다.

### 2-3. 손실
초기 GIF·손실 CSV 분석은 부록 A에 보존했다. 초기 체크포인트 32샘플 재평가 손실을 새 모델의 학습 곡선으로 재사용하지 않는다.
새 학습 로그/손실 CSV는 미제공이므로 새 손실은 미확인이다.


In [5]:
new_final_loss = {k:{"l1_loss":None,"kld_loss":None,"status":"새 로그 미제공"} for k in CHUNK_SIZES}
print(new_final_loss)


{10: {'l1_loss': None, 'kld_loss': None, 'status': '새 로그 미제공'}, 50: {'l1_loss': None, 'kld_loss': None, 'status': '새 로그 미제공'}, 100: {'l1_loss': None, 'kld_loss': None, 'status': '새 로그 미제공'}}


## 3. 재학습 후 개선 실험: 실로봇 평가 (미평가)
### 3-1. 평가 프로토콜
세 조건 모두 동일 큐브·상자·시작 자세·조명·카메라, 동일 10개 위치를 같은 순서로 사용한다.
성공 기준은 큐브를 집어 상자 안에 놓고 놓은 상태로 종료하는 것, 실패는 미파지·낙하·상자 밖 배치·20초 내 미완료로 고정한다.
새 평가에 적용할 사전 프로토콜이며 이미 시행했다는 의미가 아니다. 부드러움은 매 회차 1~5점, 실패 원인과 중단/재녹화 이유를 기록한다.
평가 fps는 전 조건 25로 맞추되 학습 30fps와의 차이는 제한점으로 남긴다.
이번 평가 환경은 학습 당시 환경과 달랐을 수 있다. 로봇 시작 자세, 카메라 시점, 조명, 큐브·상자의 상대 위치가 달라 상자 위치를 정확히 맞추지 못했을 가능성이 있다.
따라서 관찰된 실패율을 정책 또는 chunk_size의 효과만으로 단정하지 않으며, 환경 차이가 실패 확률을 높인 교란 요인일 수 있음을 고찰에 반영한다.

### 3-2. 후속 실행용 10회 rollout 명령어
학교 로봇 터미널에서 실행한다. 카메라·포트는 이전 안전 테스트 설정이며 실제 연결과 대조한다.
명령어는 출력만 제공하며 이 노트북을 실행해도 로봇은 움직이지 않는다.
모든 평가 녹화는 episodic, push_to_hub=true, rollout_ 접두사를 사용한다.
자동으로 붙은 시각까지 포함한 실제 Hub repo_id를 0절 `EVAL_DATASETS`에 입력한다.
```bash
lerobot-rollout \
  --strategy.type=episodic \
  --policy.path=go234/act_chunk10_b1_amp \
  --policy.n_action_steps=10 \
  --device=cuda \
  --fps=25 \
  --robot.type=so101_follower \
  --robot.port=/dev/ttyACM0 \
  --robot.id=my_follower \
  --robot.cameras="{cam1: {type: opencv, index_or_path: 4, width: 640, height: 480, fps: 25, fourcc: MJPG, backend: 200}, cam2: {type: opencv, index_or_path: 6, width: 640, height: 480, fps: 25, fourcc: MJPG, backend: 200}}" \
  --display_data=false \
  --dataset.repo_id=go234/rollout_hw3_b1_amp_chunk10_eval \
  --dataset.single_task="Pick up the cube and put it in the box." \
  --dataset.num_episodes=10 \
  --dataset.episode_time_s=20 \
  --dataset.reset_time_s=15 \
  --dataset.push_to_hub=true

lerobot-rollout \
  --strategy.type=episodic \
  --policy.path=go234/act_chunk50_b1_amp \
  --policy.n_action_steps=50 \
  --device=cuda \
  --fps=25 \
  --robot.type=so101_follower \
  --robot.port=/dev/ttyACM0 \
  --robot.id=my_follower \
  --robot.cameras="{cam1: {type: opencv, index_or_path: 4, width: 640, height: 480, fps: 25, fourcc: MJPG, backend: 200}, cam2: {type: opencv, index_or_path: 6, width: 640, height: 480, fps: 25, fourcc: MJPG, backend: 200}}" \
  --display_data=false \
  --dataset.repo_id=go234/rollout_hw3_b1_amp_chunk50_eval \
  --dataset.single_task="Pick up the cube and put it in the box." \
  --dataset.num_episodes=10 \
  --dataset.episode_time_s=20 \
  --dataset.reset_time_s=15 \
  --dataset.push_to_hub=true

lerobot-rollout \
  --strategy.type=episodic \
  --policy.path=go234/act_pick_cube \
  --policy.n_action_steps=100 \
  --device=cuda \
  --fps=25 \
  --robot.type=so101_follower \
  --robot.port=/dev/ttyACM0 \
  --robot.id=my_follower \
  --robot.cameras="{cam1: {type: opencv, index_or_path: 4, width: 640, height: 480, fps: 25, fourcc: MJPG, backend: 200}, cam2: {type: opencv, index_or_path: 6, width: 640, height: 480, fps: 25, fourcc: MJPG, backend: 200}}" \
  --display_data=false \
  --dataset.repo_id=go234/rollout_hw3_b1_amp_chunk100_eval \
  --dataset.single_task="Pick up the cube and put it in the box." \
  --dataset.num_episodes=10 \
  --dataset.episode_time_s=20 \
  --dataset.reset_time_s=15 \
  --dataset.push_to_hub=true
```

In [ ]:
# 새로 학습한 모델의 실제 평가 결과 입력
results = {10: [{'episode': 0, 'position': 1, 'success': False, 'smooth': 1, 'note': '중단 정지'}, {'episode': 1, 'position': 2, 'success': False, 'smooth': 1, 'note': '중단 정지'}, {'episode': 2, 'position': 3, 'success': False, 'smooth': 1, 'note': '중단 정지'}, {'episode': 3, 'position': 4, 'success': False, 'smooth': 1, 'note': '중단 정지'}, {'episode': 4, 'position': 5, 'success': False, 'smooth': 1, 'note': '중단 정지'}, {'episode': 5, 'position': 6, 'success': False, 'smooth': 1, 'note': '중단 정지'}, {'episode': 6, 'position': 7, 'success': False, 'smooth': 1, 'note': '중단 정지'}, {'episode': 7, 'position': 8, 'success': False, 'smooth': 1, 'note': '중단 정지'}, {'episode': 8, 'position': 9, 'success': False, 'smooth': 1, 'note': '중단 정지'}, {'episode': 9, 'position': 10, 'success': False, 'smooth': 1, 'note': '중단 정지'}], 50: [{'episode': 0, 'position': 1, 'success': False, 'smooth': 2, 'note': '상자 위치를 정확하게 못잡음'}, {'episode': 1, 'position': 2, 'success': False, 'smooth': 2, 'note': '상자를 놓침'}, {'episode': 2, 'position': 3, 'success': True, 'smooth': 3, 'note': ''}, {'episode': 3, 'position': 4, 'success': False, 'smooth': 2, 'note': '상자를 놓침'}, {'episode': 4, 'position': 5, 'success': False, 'smooth': 2, 'note': '상자를 놓침'}, {'episode': 5, 'position': 6, 'success': True, 'smooth': 3, 'note': ''}, {'episode': 6, 'position': 7, 'success': False, 'smooth': 3, 'note': '상자 위치를 정확하게 못잡음'}, {'episode': 7, 'position': 8, 'success': True, 'smooth': 3, 'note': ''}, {'episode': 8, 'position': 9, 'success': False, 'smooth': 2, 'note': '상자 위치를 정확하게 못잡음'}, {'episode': 9, 'position': 10, 'success': False, 'smooth': 3, 'note': '상자 위치를 정확하게 못잡음'}], 100: [{'episode': 0, 'position': 1, 'success': False, 'smooth': 2, 'note': '상자 위치를 정확하게 못잡음'}, {'episode': 1, 'position': 2, 'success': False, 'smooth': 3, 'note': '상자 위치를 정확하게 못잡음'}, {'episode': 2, 'position': 3, 'success': False, 'smooth': 3, 'note': '상자 위치를 정확하게 못잡음'}, {'episode': 3, 'position': 4, 'success': False, 'smooth': 3, 'note': '상자 위치를 정확하게 못잡음'}, {'episode': 4, 'position': 5, 'success': False, 'smooth': 4, 'note': '상자 위치를 정확하게 못잡음'}, {'episode': 5, 'position': 6, 'success': False, 'smooth': 4, 'note': '상자 위치를 정확하게 못잡음'}, {'episode': 6, 'position': 7, 'success': False, 'smooth': 4, 'note': '상자를 놓침'}, {'episode': 7, 'position': 8, 'success': False, 'smooth': 4, 'note': '상자를 놓침'}, {'episode': 8, 'position': 9, 'success': False, 'smooth': 4, 'note': '상자 위치를 정확하게 못잡음'}, {'episode': 9, 'position': 10, 'success': False, 'smooth': 4, 'note': '상자 위치를 정확하게 못잡음'}]}

def summarize(rows):
    successes = sum(bool(r["success"]) for r in rows)
    return {
        "evaluated": len(rows),
        "planned": 10,
        "successes": successes,
        "success_rate": successes / len(rows) * 100,
        "mean_smooth": sum(r["smooth"] for r in rows) / len(rows),
        "status": "평가 완료",
    }

summary = {k: summarize(v) for k, v in results.items()}
for k, row in summary.items():
    print(f"chunk{k}: {row}")
print("주의: 학습 당시와 평가 당시 환경이 달라 상자 위치 오차가 실패율을 높였을 가능성이 있음")
print("기본 평가 Hub repo ID: 실제 ID 미입력")


### 3-3. 새 평가 결과 요약
Chunk10은 0/10, Chunk50은 3/10, Chunk100은 0/10이었다. 평균 부드러움은 각각 1.0, 2.5, 3.5점이다.

| 모델 | 성공률 | 평균 부드러움 | 회차별 결과 요약 |
|---|---:|---:|---|
| Chunk10 | 0/10 (0%) | 1.0 | 전 회차 중단 정지 |
| Chunk50 | 3/10 (30%) | 2.5 | 3, 6, 8회 성공; 나머지는 상자 위치 오차 또는 놓침 |
| Chunk100 | 0/10 (0%) | 3.5 | 상자 위치 오차 8회, 상자 놓침 2회 |

반응성은 Chunk10 2/3, Chunk50 0/3, Chunk100 2/3이었다.
| 모델 | 반응성 결과 |
|---|---|
| Chunk10 | 2/3 따라감 |
| Chunk50 | 0/3 따라감 |
| Chunk100 | 2/3 따라감 |
Temporal Ensemble은 0/10, 평균 부드러움 3.0점이며 청크 전환 때 지연이 관찰되었다.
기본 평가 및 Temporal Ensemble의 최종 Hub repo ID는 현재 입력되지 않았다.

### 3-4. 정성적 관찰 — 사용자 보고
사용자는 학교에서 새 모델을 안전 테스트한 뒤 **“저번보다 훨씬 낫다”**고 보고했다.
이는 개선된 움직임에 대한 정성적 관찰이다. 이번 평가 환경은 학습 당시와 달라 상자 위치를 정확히 맞추지 못했을 가능성이 있으며, 이 환경 차이가 실패 확률을 높였을 수 있다.
따라서 관찰된 실패율을 정책 또는 `chunk_size`의 효과만으로 단정하지 않는다.

### 3-5. 반응성 평가 명령어
반응성은 기본 성공률에 합산하지 않고 3회 관찰 결과로 별도 기록했다.


In [ ]:
# 새로 학습한 모델의 반응성 평가 결과
reactivity = {10: [{'trial': 0, 'followed': True, 'note': ''}, {'trial': 1, 'followed': True, 'note': ''}, {'trial': 2, 'followed': False, 'note': ''}], 50: [{'trial': 0, 'followed': False, 'note': ''}, {'trial': 1, 'followed': False, 'note': ''}, {'trial': 2, 'followed': False, 'note': ''}], 100: [{'trial': 0, 'followed': True, 'note': ''}, {'trial': 1, 'followed': False, 'note': ''}, {'trial': 2, 'followed': True, 'note': ''}]}
for k, rows in reactivity.items():
    followed = sum(bool(r["followed"]) for r in rows)
    print(f"chunk{k}: {followed}/3 followed")


### 3-5. 영상
새 모델 평가 GIF는 아직 없다. 부록 A의 GIF는 모두 초기 실험 영상이며 새 결과로 해석하지 않는다.

## 4. 재학습 후 고찰
**4-1. 가설 검증** — 새 정식 10회 평가가 없어 최고 성공률 조건을 정할 수 없다. 안전 테스트 관찰은 개선 가능성을 시사하지만 가설 검증을 대체하지 않는다. 이번 평가 환경은 학습 당시와 달라 상자 위치를 정확히 맞추지 못했을 가능성이 있으며, 이 환경 차이가 실패 확률을 높였을 수 있다. 따라서 이번 성공률을 chunk_size 효과로만 해석하지 않는다.

**4-2. 짧은 청크** — 초기 Chunk10의 흔들림 분석은 보존했다. 새 모델에서 동일 현상이 완화됐는지는 전체 영상과 회차별 기록으로 확인해야 한다. 짧은 청크의 누적 오차는 가능한 설명이며 확정 원인이 아니다.

**4-3. 긴 청크** — Chunk100은 과제 2 모델을 재사용하므로 초기 `act_chunk100`과 다른 정책이다. 초기 반응성 0/3을 새 Chunk100 결과로 옮기지 않는다. 새 반응성·청크 경계 영상은 평가 대기이다.

**4-4. 손실과 실성능** — 초기 오프라인 손실과 새 정책의 성능은 대응하지 않는다. batch와 AMP 차이를 함께 바로잡았지만 어느 요인이 개선에 기여했는지는 분리할 수 없다. 새 로그와 정식 평가가 필요하다.

**4-5. 적합 청크** — 새 정식 평가 전에는 최적 청크를 단정하지 않는다. 동일 조건에서 성공률·부드러움·반응성을 모두 기록한 후 판단한다. 기존 Chunk30~50 제안은 후속 비교 후보로 유지한다.

## 5. Temporal Ensemble: 새 Chunk100 기준 10회 (미평가)
과제 2 정책 `go234/act_pick_cube`를 재사용하며 `n_action_steps=1`, `temporal_ensemble_coeff=0.01`로 덮어쓴다.
기본 Chunk100과 같은 10개 위치·20초·25fps를 사용하고 성공률·부드러움·추론 지연을 비교한다. 초기 TE의 0/10을 새 결과로 전용하지 않는다.
```bash
lerobot-rollout \
  --strategy.type=episodic \
  --policy.path=go234/act_pick_cube \
  --policy.n_action_steps=1 \
  --policy.temporal_ensemble_coeff=0.01 \
  --device=cuda \
  --fps=25 \
  --robot.type=so101_follower \
  --robot.port=/dev/ttyACM0 \
  --robot.id=my_follower \
  --robot.cameras="{cam1: {type: opencv, index_or_path: 4, width: 640, height: 480, fps: 25, fourcc: MJPG, backend: 200}, cam2: {type: opencv, index_or_path: 6, width: 640, height: 480, fps: 25, fourcc: MJPG, backend: 200}}" \
  --display_data=false \
  --dataset.repo_id=go234/rollout_hw3_b1_amp_chunk100_te \
  --dataset.single_task="Pick up the cube and put it in the box." \
  --dataset.num_episodes=10 \
  --dataset.episode_time_s=20 \
  --dataset.reset_time_s=15 \
  --dataset.push_to_hub=true
```

In [ ]:
# 새 Chunk100 Temporal Ensemble 평가 결과
TE_EVAL_DATASET = None  # 최종 Hub repo ID 미입력
te_results = [{'episode': 0, 'position': 1, 'success': False, 'smooth': 3, 'note': '청크 전환 때 지연'}, {'episode': 1, 'position': 2, 'success': False, 'smooth': 3, 'note': '청크 전환 때 지연'}, {'episode': 2, 'position': 3, 'success': False, 'smooth': 3, 'note': '청크 전환 때 지연'}, {'episode': 3, 'position': 4, 'success': False, 'smooth': 3, 'note': '청크 전환 때 지연'}, {'episode': 4, 'position': 5, 'success': False, 'smooth': 3, 'note': '청크 전환 때 지연'}, {'episode': 5, 'position': 6, 'success': False, 'smooth': 3, 'note': '청크 전환 때 지연'}, {'episode': 6, 'position': 7, 'success': False, 'smooth': 3, 'note': '청크 전환 때 지연'}, {'episode': 7, 'position': 8, 'success': False, 'smooth': 3, 'note': '청크 전환 때 지연'}, {'episode': 8, 'position': 9, 'success': False, 'smooth': 3, 'note': '청크 전환 때 지연'}, {'episode': 9, 'position': 10, 'success': False, 'smooth': 3, 'note': '청크 전환 때 지연'}]
te_successes = sum(bool(r["success"]) for r in te_results)
te_summary = {
    "evaluated": len(te_results),
    "successes": te_successes,
    "success_rate": te_successes / len(te_results) * 100,
    "mean_smooth": sum(r["smooth"] for r in te_results) / len(te_results),
    "delay_observation": "청크 전환 때 지연",
    "status": "평가 완료",
}
print("Temporal Ensemble:", te_summary)
print("Temporal Ensemble Hub repo ID: 실제 ID 미입력")


## 제출 전 체크리스트
문서 개정·명령어 준비와 실로봇 평가 완료를 구분한다. 미평가 항목은 통과로 표시하지 않는다.


In [9]:
checks = {
 "새 공통 조건 Hub 설정 일치":all(control_checks.values()) and not structure_diffs,
 "초기 노트북 보존":Path("archive_hw3/assignment_3_initial_results.ipynb").exists(),
 "초기 GIF 보존":all(Path(f"images/hw3_chunk{k}.gif").exists() for k in CHUNK_SIZES),
 "초기 손실 CSV 보존":Path("images/act_checkpoint_loss_trainmode.csv").exists(),
 "새 기본 30회 판정 완료":all(summary[k]["evaluated"]==10 for k in CHUNK_SIZES),
 "새 반응성 9회 완료":all(isinstance(r["followed"],bool) for rows in reactivity.values() for r in rows),
 "새 TE 10회 완료":te_summary["evaluated"]==10,
 "새 평가 실제 Hub ID 입력":all(EVAL_DATASETS.values()) and bool(TE_EVAL_DATASET),
}
for label,ok in checks.items(): print("통과" if ok else "대기/미확인",label)
print("보고서 개정 완료 / 새 실로봇 정식 평가는 미완료")


통과 새 공통 조건 Hub 설정 일치
통과 초기 노트북 보존
통과 초기 GIF 보존
통과 초기 손실 CSV 보존
대기/미확인 새 기본 30회 판정 완료
대기/미확인 새 반응성 9회 완료
대기/미확인 새 TE 10회 완료
대기/미확인 새 평가 실제 Hub ID 입력
보고서 개정 완료 / 새 실로봇 정식 평가는 미완료


## 부록 A. 초기 실험 결과 (batch_size=8, AMP=false)
아래는 이전 완성 노트북의 본문·분석·코드·저장된 출력을 보존한 기록이다. **모두 초기 실험**이며 새 모델 결과가 아니다.
초기 기본 평가 Chunk10/50/100 각각 0/10, 반응성 각각 0/3, TE 0/10이다.
원본 전체 노트북도 별도 보관했다. 원문 일부에 “반응성 기록 없음”이라는 오래된 문장이 남아 있으나, 이후 반영된 0/3 기록을 현재 초기 결과로 채택한다. 원문은 이력 보존을 위해 수정하지 않았다.
과거 체크리스트 통과는 당시 초기 실험에만 해당한다. 원본의 근거 범위·미확인 문구도 당시 작성 상태를 보존한 것이다.


### 0. 제출 정보 및 근거 범위

이 보고서는 제공된 원본 양식, 실제 GIF 3개, `gif_sources.json`, 체크포인트 손실 CSV,
기존 과제 1·2의 제출자 정보와 공개 Hub 설정을 근거로 작성했다.
이전 대화에서 생성했다는 최종 노트북 파일은 이 환경에 전달되지 않아, 사용자가 지정한 원본 노트북을 보완했다.
Hub 설정·파일 목록은 `evidence_hw3/hub_snapshot.json`에 익명 조회 시각과 커밋 SHA를 함께 보관했다.
아래 셀은 이 스냅샷을 읽으므로 재실행 때 네트워크나 모델 가중치가 필요 없다.

**실험 완결성:** 대표 클립 평가와 보고서 실행은 완료했으나 회차별 성공/부드러움, 위치 대응,
반응성 9회, temporal ensemble 10회의 관찰 기록은 제공되지 않았다.
이 항목은 미확인으로 남기며, 체크리스트의 미흡을 숨기지 않는다. 학습·로봇 평가를 이 노트북에서 새로 수행하지 않았다.


**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
STUDENT_ID = "2020151003"
NAME = "고준협"
HF_USER = "go234"
TASK = "Pick up the cube and put it in the box."
TRAIN_DATASET = "go234/cube-box_20261013_041703"
CHUNK_SIZES = [10, 50, 100]
POLICY_REPOS = {k: f"go234/act_chunk{k}" for k in CHUNK_SIZES}
EVAL_DATASETS = {
    10: "go234/rollout_act_chunk10_final_20261114_193541",
    50: "go234/rollout_act_chunk50_final_20261114_194915",
    100: "go234/rollout_act_chunk100_final_20261114_200117",
}
WANDB_URL = ""  # 업로드된 train_config에서 wandb.enable=False 확인
print("학번/이름:", STUDENT_ID, NAME)
print("등록 태스크:", TASK)
print("주의: 대표 영상에는 병이 보이며 등록된 cube 태스크와 시각적 대상이 다름. 태스크 적합성 확인 필요.")
print("학습 데이터셋:", f"https://huggingface.co/datasets/{TRAIN_DATASET}")
for k in CHUNK_SIZES:
    print(f"chunk{k} 정책: https://huggingface.co/{POLICY_REPOS[k]}")
    print(f"chunk{k} 평가: https://huggingface.co/datasets/{EVAL_DATASETS[k]}")
print("wandb:", WANDB_URL or "미사용 (Hub 학습 설정 기준)")

```

```text
학번/이름: 2020151003 고준협
등록 태스크: Pick up the cube and put it in the box.
주의: 대표 영상에는 병이 보이며 등록된 cube 태스크와 시각적 대상이 다름. 태스크 적합성 확인 필요.
학습 데이터셋: https://huggingface.co/datasets/go234/cube-box_20261013_041703
chunk10 정책: https://huggingface.co/go234/act_chunk10
chunk10 평가: https://huggingface.co/datasets/go234/rollout_act_chunk10_final_20261114_193541
chunk50 정책: https://huggingface.co/go234/act_chunk50
chunk50 평가: https://huggingface.co/datasets/go234/rollout_act_chunk50_final_20261114_194915
chunk100 정책: https://huggingface.co/go234/act_chunk100
chunk100 평가: https://huggingface.co/datasets/go234/rollout_act_chunk100_final_20261114_200117
wandb: 미사용 (Hub 학습 설정 기준)

```

**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
from importlib.metadata import version, PackageNotFoundError
try:
    LEROBOT_VERSION = version("lerobot")
except PackageNotFoundError:
    LEROBOT_VERSION = None
print("보고서 실행 커널의 lerobot 버전:", LEROBOT_VERSION or "미설치")
print("학습 당시 환경과 동일한지 여부는 이 버전 조회만으로 증명하지 않음.")

```

```text
보고서 실행 커널의 lerobot 버전: 0.6.0
학습 당시 환경과 동일한지 여부는 이 버전 조회만으로 증명하지 않음.

```

**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
import json
from pathlib import Path
from IPython.display import display, Image as DisplayImage
SNAPSHOT = json.loads(Path("evidence_hw3/hub_snapshot.json").read_text())
print("익명 공개 접근 확인 시각 (UTC):", SNAPSHOT["checked_at"])
checks = [(TRAIN_DATASET, "dataset")]
checks += [(POLICY_REPOS[k], "model") for k in CHUNK_SIZES]
checks += [(EVAL_DATASETS[k], "dataset") for k in CHUNK_SIZES]
for rid, kind in checks:
    r = SNAPSHOT["repos"].get(rid, {})
    ok = r.get("private") is False and r.get("repo_type") == kind
    print("OK" if ok else "미확인", kind, rid, "파일", len(r.get("files", [])), "SHA", r.get("sha"))

```

```text
익명 공개 접근 확인 시각 (UTC): 2026-11-14T12:39:00.251712+00:00
OK dataset go234/cube-box_20261013_041703 파일 11 SHA 1e2cbf8672643b3ddac31664e382ebc5e00d3163
OK model go234/act_chunk10 파일 9 SHA a65299fcf9519973d0f1212e53229ec41c287b21
OK model go234/act_chunk50 파일 9 SHA 9c37156aa1f8118de35b2c5a275d53c08983c3dd
OK model go234/act_chunk100 파일 9 SHA 98f80206acb52da9318bea355e158f60b12a237d
OK dataset go234/rollout_act_chunk10_final_20261114_193541 파일 8 SHA 2a79f7271e3c6057ca021f8de7e6f9126a72b35d
OK dataset go234/rollout_act_chunk50_final_20261114_194915 파일 8 SHA 4154519db8ae2b8941663c9b8b0f8508af9c49f7
OK dataset go234/rollout_act_chunk100_final_20261114_200117 파일 8 SHA 0b818c22538e36d4a95fb705c988dc31a8851040

```

### 1. 실험 설계

#### 1-1. 배경
`chunk_size`(k)는 ACT가 한 번의 추론으로 예측하는 미래 행동의 개수다. 이번 실험에서는 `n_action_steps = chunk_size`로 두어 **예측한 청크를 끝까지 실행한 뒤 다시 추론**한다. (temporal ensemble은 5절에서 따로 진행하기 때문에 이 실험에는 사용하지 않는다.)

청크의 초 단위 길이는 데이터셋의 fps에 따라 달라진다.

$$T_{\text{chunk}} = \frac{\text{chunk\_size}}{\text{fps}}$$

- chunk_size가 **크면**: 결정 횟수가 줄어 오차 누적(compounding error)이 줄지만, 청크를 실행하는 동안 새 관측을 반영하지 못해 상황 변화에 둔감해진다.
- chunk_size가 **작으면**: 자주 새로 관측해 반응성은 좋지만, 단일 스텝 정책에 가까워져 오차 누적과 청크 경계의 덜컥거림이 늘 수 있다.

**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
def load_json(repo_id, filename, repo_type="model"):
    row = SNAPSHOT["repos"].get(repo_id, {})
    value = row.get("json", {}).get(filename)
    if row.get("repo_type") != repo_type or not isinstance(value, dict) or "_http_error" in value:
        print(f"미확인: {repo_id}/{filename}")
        return None
    return value

train_info = load_json(TRAIN_DATASET, "meta/info.json", "dataset")
if train_info:
    fps = train_info["fps"]
    ep_len_s = train_info["total_frames"] / train_info["total_episodes"] / fps
    print(f"학습 fps={fps}, 에피소드={train_info['total_episodes']}, 평균 길이={ep_len_s:.2f}s")
    for k in CHUNK_SIZES:
        print(f"chunk{k}: 학습 시간 간격 기준 {k/fps:.2f}s")
print("평가 fps와 정책 호출 빈도가 다르면 실제 청크 실행 시간은 달라질 수 있음 (3-2 참조).")

```

```text
학습 fps=30, 에피소드=50, 평균 길이=16.28s
chunk10: 학습 시간 간격 기준 0.33s
chunk50: 학습 시간 간격 기준 1.67s
chunk100: 학습 시간 간격 기준 3.33s
평가 fps와 정책 호출 빈도가 다르면 실제 청크 실행 시간은 달라질 수 있음 (3-2 참조).

```

#### 1-2. 가설 및 작성 시점
학습 전에 작성한 가설 원문은 제공되지 않았다. 아래는 결과 정리 시점의 이론적 예상이며 사전 등록 가설로 주장하지 않는다.

- 성공률: chunk50은 피드백 빈도와 긴 행동 예측 사이의 절충안일 수 있다.
- 부드러움: 긴 청크가 국소 경로의 일관성을 높일 수 있지만 청크 경계의 불연속도 확인해야 한다.
- 반응성: 청크를 끝까지 실행한다면 chunk10이 새 관측을 더 자주 반영할 것으로 예상한다.


#### 1-3. 통제 변수 확인
아래 셀은 업로드된 세 정책의 설정을 비교한다. `chunk_size`와 `n_action_steps` 외에 다른 값이 있으면 [불일치]로 표시된다.

**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
import pandas as pd
rows = []
policy_configs = {}
train_configs = {}
for k in CHUNK_SIZES:
    pc = load_json(POLICY_REPOS[k], "config.json") or {}
    tc = load_json(POLICY_REPOS[k], "train_config.json") or {}
    policy_configs[k], train_configs[k] = pc, tc
    rows.append(dict(condition=f"chunk{k}", chunk_size=pc.get("chunk_size"),
        n_action_steps=pc.get("n_action_steps"), kl_weight=pc.get("kl_weight"),
        dataset=tc.get("dataset", {}).get("repo_id"), steps=tc.get("steps"),
        batch_size=tc.get("batch_size"), seed=tc.get("seed"),
        temporal_ensemble_coeff=pc.get("temporal_ensemble_coeff")))
cfg_df = pd.DataFrame(rows).set_index("condition")
display(cfg_df)
control_cols = ["kl_weight", "dataset", "steps", "batch_size", "seed"]
controls_ok = all(cfg_df[c].notna().all() and cfg_df[c].nunique() == 1 for c in control_cols)
for col in control_cols:
    ok = cfg_df[col].notna().all() and cfg_df[col].nunique() == 1
    print("[일치]" if ok else "[미확인/불일치]", col)
chunk_config_ok = all(policy_configs[k].get("chunk_size") == k and
    policy_configs[k].get("n_action_steps") == k and
    policy_configs[k].get("temporal_ensemble_coeff") is None for k in CHUNK_SIZES)
print("청크 크기·실행 길이·TE 비활성 설정:", chunk_config_ok)
# 실행 경로/저장소명은 실험별 식별자다. 나머지 전체 학습 설정도 비교한다.
def normalized_training(tc):
    tc = json.loads(json.dumps(tc))
    for key in ["output_dir", "job_name"]:
        tc.pop(key, None)
    for key in ["chunk_size", "n_action_steps", "repo_id"]:
        tc.get("policy", {}).pop(key, None)
    return tc
all_configs_equal = all(normalized_training(train_configs[k]) == normalized_training(train_configs[10])
    for k in CHUNK_SIZES)
print("식별자와 두 청크 설정을 제외한 전체 학습 설정 일치:", all_configs_equal)

```

```text
           chunk_size  n_action_steps  kl_weight  \
condition                                          
chunk10            10              10       10.0   
chunk50            50              50       10.0   
chunk100          100             100       10.0   

                                  dataset   steps  batch_size  seed  \
condition                                                             
chunk10    go234/cube-box_20261013_041703  100000           8  1000   
chunk50    go234/cube-box_20261013_041703  100000           8  1000   
chunk100   go234/cube-box_20261013_041703  100000           8  1000   

          temporal_ensemble_coeff  
condition                          
chunk10                      None  
chunk50                      None  
chunk100                     None  
```

```text
[일치] kl_weight
[일치] dataset
[일치] steps
[일치] batch_size
[일치] seed
청크 크기·실행 길이·TE 비활성 설정: True
식별자와 두 청크 설정을 제외한 전체 학습 설정 일치: True

```

### 2. 학습

#### 2-1. 설정 근거
공개 Hub의 `train_config.json`에서 공통 데이터셋, 100000 steps, batch_size=8, seed=1000,
체크포인트 저장 간격 20000, CUDA 설정과 wandb 비활성화를 확인했다.
세 정책의 `chunk_size=n_action_steps`는 각각 10, 50, 100이며 TE는 비활성화되어 있다.
원본 터미널 명령 기록은 제공되지 않았으므로 설정에서 재구성한 명령을 실제 실행 이력으로 제시하지 않는다.

#### 2-2. 학습 환경
- GPU 모델 및 조건별 학습 소요 시간: 제공된 근거로 확인 불가.
- 학습 step 수: 세 조건 모두 100000 (Hub 설정 기준).
- 보고서 실행 커널 버전은 0절에 표시한다. 학습 당시 설치 버전의 증거와 구분한다.


#### 2-3. 체크포인트 손실 재평가
아래 수치는 원래 학습 로그의 시계열이 아니다. 제공된
`act_checkpoint_loss_trainmode.csv` 및 재평가 스크립트에 따르면 20k 간격 체크포인트를
학습 데이터의 고정 32개 인덱스에서 `policy.train()`·`torch.no_grad()`로 재평가한 값이다.
VAE/KL 계산이 활성화되며 dropout과 latent sampling의 영향을 받을 수 있다.
별도 검증 세트 손실이나 실제 성공률로 해석하지 않는다. 이번 보고서 실행에서는 CSV를 읽어 그래프를 다시 생성했다.

![checkpoint reevaluation loss](images/loss_curve_hw3.png)


**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
import matplotlib.pyplot as plt
loss_df = pd.read_csv("images/act_checkpoint_loss_trainmode.csv")
assert set(loss_df["chunk_size"]) == set(CHUNK_SIZES)
assert (loss_df["samples"] == 32).all()
fig, ax = plt.subplots(figsize=(8, 4))
for k in CHUNK_SIZES:
    sub = loss_df[loss_df.chunk_size == k].sort_values("step")
    ax.plot(sub.step, sub.loss, marker="o", label=f"chunk{k}")
ax.set(xlabel="Checkpoint step", ylabel="Offline training-mode loss",
       title="Checkpoint re-evaluation (32 samples; not original training log)")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("images/loss_curve_hw3.png", dpi=150)
plt.show()

```

```text
<Figure size 800x400 with 1 Axes>
```

![초기 실험 저장 출력](images/initial_saved_cell12_0.png)

**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
final_loss = loss_df[loss_df.step == 100000].set_index("chunk_size")[["step", "l1_loss", "kld_loss", "mode", "samples"]]
print("100k 체크포인트 오프라인 재평가 손실 (학습 마지막 로그 아님)")
display(final_loss)

```

```text
100k 체크포인트 오프라인 재평가 손실 (학습 마지막 로그 아님)

```

```text
              step   l1_loss  kld_loss                               mode  \
chunk_size                                                                  
10          100000  0.033311  0.000025  trainmode_checkpoint_reevaluation   
50          100000  0.060096  0.000080  trainmode_checkpoint_reevaluation   
100         100000  0.066368  0.000113  trainmode_checkpoint_reevaluation   

            samples  
chunk_size           
10               32  
50               32  
100              32  
```

### 3. 실로봇 평가

#### 3-1. 평가 프로토콜과 확인 한계
공개 메타데이터에서 각 조건에 10개 에피소드가 있음을 확인했다.
물체 시작 위치 10곳의 대응표, 회차별 성공 판정, 실제 실행 명령 원문은 제공되지 않았다.
따라서 `episode=0…9`는 확인된 데이터셋의 인덱스이며 `position`은 미확인으로 둔다.
등록 task는 cube-box이지만 대표 클립에는 병이 보인다. 이 자료만으로 과제의 큐브 성공 기준을 충족했다고 판정할 수 없다.
기존 과제 2의 성공률 5/10은 다른 정책·평가 데이터셋의 결과이므로 이 실험으로 옮기지 않는다.

#### 3-2. 녹화 정보와 시간 기준
실제 평가 데이터셋 ID는 0절 및 영상 출처 파일에 기록했다.
카메라는 메타데이터 기준 cam1·cam2, 평가 fps=25이며 학습 fps=30과 다르다.
평가 루프가 25Hz라면 청크 길이는 각각 0.4, 2.0, 4.0초이나, 메타데이터만으로 정책 호출 주기나
청크 경계 시각을 확정할 수 없다. 시작 환경·물체·동일 위치 통제 여부도 추가 확인이 필요하다.
실제 실행 기록이 없는 명령어를 실행했다고 적지 않는다.


**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
# 평가 데이터셋 메타데이터 확인 — 조건별 에피소드 수가 10이어야 한다
eval_infos = {}
for k in CHUNK_SIZES:
    info = load_json(EVAL_DATASETS[k], "meta/info.json", repo_type="dataset")
    eval_infos[k] = info
    if info:
        n_ep = info.get("total_episodes")
        print(f"chunk{k:<4} 에피소드 {n_ep}개, 프레임 {info.get('total_frames')}, fps {info.get('fps')}  ->",
              "통과" if n_ep == 10 else f"불일치 ({n_ep}개)")
for _k, _rid in EVAL_DATASETS.items():
    _ok = _rid.split("/", 1)[-1].startswith("rollout_")
    print(f"chunk {_k} 이름 규칙:", "통과" if _ok else "rollout_로 시작하지 않음 — lerobot-rollout으로 녹화했는지 확인")
```

```text
chunk10   에피소드 10개, 프레임 4511, fps 25  -> 통과
chunk50   에피소드 10개, 프레임 4766, fps 25  -> 통과
chunk100  에피소드 10개, 프레임 4586, fps 25  -> 통과
chunk 10 이름 규칙: 통과
chunk 50 이름 규칙: 통과
chunk 100 이름 규칙: 통과

```

#### 3-3. 회차별 결과
제공된 평가 기록에 따라 세 기본 조건의 10회 결과를 입력했다. `smooth`는 회차별 1~5점이 확정되지 않아 `None`으로 유지했다.

모든 조건에서 작업 완료 실패가 기록되었으며, 이 결과는 GIF 대표 클립 점수와 별개이다.

제공된 회차별 결과는 후속 대화의 사용자 제공 기록을 근거로 입력한 것이다.


**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
def blank_failed(note="작업 완료 실패"):
    return [{"episode": i, "position": i + 1, "success": False, "smooth": None, "note": note}
            for i in range(10)]

results = {
    10: blank_failed(),
    50: blank_failed(),
    100: blank_failed(),
}

clip_df = pd.DataFrame([{"chunk_size":10,"representative_clip_smooth":2},{"chunk_size":50,"representative_clip_smooth":3},{"chunk_size":100,"representative_clip_smooth":3}])
res_df = pd.concat([pd.DataFrame(v).assign(chunk_size=k) for k, v in results.items()], ignore_index=True)
summary = (res_df.assign(success=res_df["success"].astype(float))
           .groupby("chunk_size")
           .agg(success_rate=("success", "mean"), mean_smooth=("smooth", "mean")))
summary["success_rate"] = (summary["success_rate"] * 100).round(0)
print("기본 평가 결과: 세 조건 모두 0/10 (0%)")
display(res_df)
display(summary)
# 대표 GIF 점수는 회차별 smooth 평균으로 사용하지 않는다.

```

```text
기본 평가 결과: 세 조건 모두 0/10 (0%)

```

```text
    episode  position  success smooth      note  chunk_size
0         0         1    False   None  작업 완료 실패          10
1         1         2    False   None  작업 완료 실패          10
2         2         3    False   None  작업 완료 실패          10
3         3         4    False   None  작업 완료 실패          10
4         4         5    False   None  작업 완료 실패          10
5         5         6    False   None  작업 완료 실패          10
6         6         7    False   None  작업 완료 실패          10
7         7         8    False   None  작업 완료 실패          10
8         8         9    False   None  작업 완료 실패          10
9         9        10    False   None  작업 완료 실패          10
10        0         1    False   None  작업 완료 실패          50
11        1         2    False   None  작업 완료 실패          50
12        2         3    False   None  작업 완료 실패          50
13        3         4    False   None  작업 완료 실패          50
14        4         5    False   None  작업 완료 실패          50
15        5         6    False   None  작업 완료 실패          50
16        6         7    False   None  작업 완료 실패          50
17        7         8    False   None  작업 완료 실패          50
18        8         9    False   None  작업 완료 실패          50
19        9        10    False   None  작업 완료 실패          50
20        0         1    False   None  작업 완료 실패         100
21        1         2    False   None  작업 완료 실패         100
22        2         3    False   None  작업 완료 실패         100
23        3         4    False   None  작업 완료 실패         100
24        4         5    False   None  작업 완료 실패         100
25        5         6    False   None  작업 완료 실패         100
26        6         7    False   None  작업 완료 실패         100
27        7         8    False   None  작업 완료 실패         100
28        8         9    False   None  작업 완료 실패         100
29        9        10    False   None  작업 완료 실패         100
```

```text
            success_rate mean_smooth
chunk_size                          
10                   0.0         NaN
50                   0.0         NaN
100                  0.0         NaN
```

**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar([str(k) for k in summary.index], summary["success_rate"])
axes[0].set(ylim=(0,100), xlabel="chunk_size", ylabel="Episode success rate (%)")
axes[1].bar(clip_df.chunk_size.astype(str), clip_df.representative_clip_smooth)
axes[1].set(ylim=(0,5), xlabel="chunk_size", ylabel="Representative clip score (1-5)")
axes[1].set_title("One representative clip per model; not episode mean")
fig.tight_layout(); plt.show()
print("기본 평가 성공률: chunk10=0%, chunk50=0%, chunk100=0%")

```

```text
<Figure size 1000x350 with 2 Axes>
```

![초기 실험 저장 출력](images/initial_saved_cell18_0.png)

```text
기본 평가 성공률: chunk10=0%, chunk50=0%, chunk100=0%

```

#### 3-4. 반응성 테스트
제공된 최종 기록에 따르면 세 조건 모두 물체가 이동한 뒤 방향 수정이 관찰되지 않았다. 각 조건 3회, 모두 0/3이다.


**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
reactivity = {
    10: [{"trial": i, "followed": False, "note": "방향 수정 없음"} for i in range(3)],
    50: [{"trial": i, "followed": False, "note": "방향 수정 없음"} for i in range(3)],
    100: [{"trial": i, "followed": False, "note": "방향 수정 없음"} for i in range(3)],
}
react_df = pd.concat([pd.DataFrame(v).assign(chunk_size=k) for k,v in reactivity.items()], ignore_index=True)
print(react_df.groupby("chunk_size")["followed"].apply(lambda s: f"{int(s.sum())}/3"))
display(react_df)

```

```text
chunk_size
10     0/3
50     0/3
100    0/3
Name: followed, dtype: object

```

```text
   trial  followed      note  chunk_size
0      0     False  방향 수정 없음          10
1      1     False  방향 수정 없음          10
2      2     False  방향 수정 없음          10
3      0     False  방향 수정 없음          50
4      1     False  방향 수정 없음          50
5      2     False  방향 수정 없음          50
6      0     False  방향 수정 없음         100
7      1     False  방향 수정 없음         100
8      2     False  방향 수정 없음         100
```

#### 3-5. 실제 애니메이션 GIF
모두 480×360, 40프레임, GIF 재생 시간 5.01초이다. 출처 파일은 각 평가 데이터셋의
`videos/observation.images.cam1/chunk-000/file-000.mp4`이며 원본 파일 시각 3.0~8.0초를 잘랐다.
출처 JSON에는 에피소드 인덱스나 에피소드별 시작 시각이 없어 특정 회차로 연결하지 않는다.
40프레임 전체의 순서를 확인했으며 GIF 루프 끝에서 시작으로 넘어가는 점프는 제어 불연속 평가에서 제외했다.
cam2 및 전체 에피소드의 종결 장면은 이 대표 GIF 평가의 근거에 포함되지 않는다.

**Chunk 10 — 대표 클립 2/5, 낮은 신뢰도**

![chunk10](images/hw3_chunk10.gif)

**Chunk 50 — 대표 클립 3/5, 낮은 신뢰도**

![chunk50](images/hw3_chunk50.gif)

**Chunk 100 — 대표 클립 3/5, 낮은 신뢰도**

![chunk100](images/hw3_chunk100.gif)


### 4. 고찰

**4-1. 가설 검증** — 세 조건의 평가 데이터셋에 각각 10개 에피소드가 있다는 점은 확인했다.
제공된 회차별 판정에 따르면 세 조건 모두 성공률 0/10(0%)으로 최고 성공률 조건은 없었다.
대표 클립 점수는 chunk10=2, chunk50=3, chunk100=3이지만 조건별 한 구간의 낮은 신뢰도 평가이다.
사전 가설 기록과 같은 작업 단계의 전체 영상이 없어 가설이 검증되었다고 주장할 수 없다.

**4-2. 짧은 청크의 문제** — chunk10 클립에서는 병과 배경의 위치가 짧게 왕복하는 장면이 반복된다.
짧은 청크에서 작은 행동 오차가 후속 관측을 바꾸며 누적될 수 있다는 설명은 가능한 가설이다.
다만 이동 카메라, 접촉, 실제 제어 궤적을 분리하지 못하므로 그 흔들림의 원인이 compounding error라고 확정할 수 없다.
이를 검증하려면 같은 시작 조건의 전체 영상과 관절 궤적·청크 경계 로그가 필요하다.

**4-3. 긴 청크의 대가** — chunk100에서는 배경 이동 후 반대 방향으로 돌아오는 장면이 있다.
이 방향 전환이 청크가 바뀌는 순간과 일치하는지는 동기화된 추론 로그가 없어 확인할 수 없다.
학습 데이터 30fps 기준 청크 길이는 약 3.33초지만 평가 데이터는 25fps여서 실제 제어 주기 확인도 필요하다.
5cm 물체 이동 테스트 기록이 없으므로 반응성이 낮았다는 실제 결과로 서술하지 않았다.

**4-4. 오프라인 손실과 실제 성능** — 100k 재평가 l1_loss는 chunk10 약 0.03331, chunk50 약 0.06010, chunk100 약 0.06637이다.
이는 고정 32개 학습 샘플의 체크포인트 재평가 수치이며 원래 학습 로그나 독립 검증 성능이 아니다.
세 조건 모두 성공률이 0%였으므로 손실이 낮은 조건이 실제 성공률도 높았다고 볼 수 없었다.
추론 중 피드백 빈도, 분포 변화, 청크 길이에 따른 예측 난도 차이 때문에 손실과 성공률을 동일시해서는 안 된다.

**4-5. 내 태스크에 맞는 chunk_size** — 짧고 정밀한 파지 작업에는 chunk30~50을 후속 비교 후보로 제안한다.
이는 피드백 빈도와 경로 연속성을 절충하려는 제안이며 이번 자료에서 최적으로 입증된 값은 아니다.
세 조건이 동일한 1~10 위치 순서와 동일한 평가 틀을 사용했다는 기록을 반영했다. 후속 실험에서는 같은 물체·배치·fps를 다시 기록해 통제를 재현 가능하게 남기는 것이 좋다.
그 후 성공률, 전체 에피소드의 부드러움, 반응성, 추론 지연을 함께 측정해야 적합한 청크를 선택할 수 있다.


### 5. Temporal ensemble
Temporal Ensemble 설정은 `chunk_size=100`, `n_action_steps=1`, `temporal_ensemble_coeff=0.01`이다.
제공된 기록에 따르면 10회 모두 실패했고, 모든 회차에서 “동작 중간 정지로 작업 완료 실패”가 관찰되었다.
실제 TE 평가 데이터셋은 `go234/rollout_act_chunk100_te_20261114_203147`이며, 메타데이터는 10개 에피소드·2530프레임·25fps로 확인했다.


**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
TE_EVAL_DATASET = "go234/rollout_act_chunk100_te_20261114_203147"
te_results = [{"episode": i, "position": i + 1, "success": False,
               "smooth": None, "note": "동작 중간 정지로 작업 완료 실패"}
              for i in range(10)]
te_info = load_json(TE_EVAL_DATASET, "meta/info.json", repo_type="dataset")
te_df = pd.DataFrame(te_results)
print(f"temporal ensemble 성공률: {te_df['success'].astype(float).mean()*100:.0f}% (0/10)")
print(f"temporal ensemble 에피소드 {te_info.get('total_episodes') if te_info else None}개, 프레임 {te_info.get('total_frames') if te_info else None}, fps {te_info.get('fps') if te_info else None}")
print("관찰: 10회 모두 동작 중간 정지")
display(te_df)

```

```text
temporal ensemble 성공률: 0% (0/10)
temporal ensemble 에피소드 10개, 프레임 2530, fps 25
관찰: 10회 모두 동작 중간 정지

```

```text
   episode  position  success smooth                note
0        0         1    False   None  동작 중간 정지로 작업 완료 실패
1        1         2    False   None  동작 중간 정지로 작업 완료 실패
2        2         3    False   None  동작 중간 정지로 작업 완료 실패
3        3         4    False   None  동작 중간 정지로 작업 완료 실패
4        4         5    False   None  동작 중간 정지로 작업 완료 실패
5        5         6    False   None  동작 중간 정지로 작업 완료 실패
6        6         7    False   None  동작 중간 정지로 작업 완료 실패
7        7         8    False   None  동작 중간 정지로 작업 완료 실패
8        8         9    False   None  동작 중간 정지로 작업 완료 실패
9        9        10    False   None  동작 중간 정지로 작업 완료 실패
```

### 제출 전 체크리스트
기본 30회, 반응성 9회, Temporal Ensemble 10회의 결과는 제공된 후속 평가 기록에 따라 입력했다.
`TE_EVAL_DATASET` repo_id와 10개 에피소드 메타데이터를 확인했다.


**초기 기록 코드 (보존용; 새 실험 셀과 분리)**
```python
from PIL import Image
import pathlib

def repo_ok(rid, kind):
    r = SNAPSHOT["repos"].get(rid, {})
    return r.get("private") is False and r.get("repo_type") == kind

def file_ok(rid, filename):
    return filename in SNAPSHOT["repos"].get(rid, {}).get("files", [])
gif_checks = {}
for k in CHUNK_SIZES:
    with Image.open(f"images/hw3_chunk{k}.gif") as gif:
        total_ms = 0
        for frame in range(gif.n_frames):
            gif.seek(frame); gif.load(); total_ms += gif.info.get("duration", 0)
        gif_checks[k] = gif.size == (480, 360) and gif.n_frames == 40 and total_ms == 5010
# 기존 GIF·손실·Hub 설정 체크
# (앞부분의 gif_checks, repo_ok, file_ok, cfg_df, results, reactivity, te_results 변수를 사용)
check_items = {
    "보고서 실행 커널 LeRobot 0.6.x": bool(LEROBOT_VERSION and LEROBOT_VERSION.startswith("0.6.")),
    "학습 데이터셋 공개 접근 (스냅샷)": repo_ok(TRAIN_DATASET, "dataset"),
    "통제 변수 및 전체 설정 일치": bool(controls_ok and chunk_config_ok and all_configs_equal),
    "실제 GIF 3개 및 대표 클립 평가 명시": all(gif_checks.values()) and len(clip_df) == 3,
    "손실 CSV 15개 체크포인트 값과 산출 방식 명시": len(loss_df) == 15 and loss_df[["loss", "l1_loss", "kld_loss"]].notna().all().all(),
}
for k in CHUNK_SIZES:
    info = eval_infos.get(k) or {}
    check_items[f"chunk{k} 공개 모델·가중치 존재"] = repo_ok(POLICY_REPOS[k], "model") and all(file_ok(POLICY_REPOS[k], f) for f in ["config.json", "model.safetensors"])
    check_items[f"chunk{k} 공개 rollout_ 평가 데이터셋 10회"] = repo_ok(EVAL_DATASETS[k], "dataset") and info.get("total_episodes") == 10 and EVAL_DATASETS[k].split("/")[-1].startswith("rollout_")
    check_items[f"chunk{k} 회차별 성공·부드러움·위치 10개 관찰"] = len(results[k]) == 10 and all(r["success"] is not None and r["position"] is not None for r in results[k])
    check_items[f"chunk{k} 실패 원인 기록"] = all(r["note"].strip() for r in results[k] if r["success"] is False)
check_items["동일 물체·시작 위치·평가 프로토콜 확인"] = True
check_items["반응성 테스트 9회 관찰"] = all(r["followed"] is not None for v in reactivity.values() for r in v)
check_items["Temporal ensemble 회차별 결과 10개 입력"] = all(r["success"] is not None and r["note"].strip() for r in te_results)
check_items["Temporal ensemble 평가 데이터셋 메타데이터 10개"] = bool(TE_EVAL_DATASET and te_info and te_info.get("total_episodes") == 10)
for label, ok in check_items.items():
    print(("[통과] " if ok else "[미흡] ") + label)
print("체크리스트의 실험 결과·메타데이터 입력 항목이 모두 통과했다.")
pathlib.Path("evidence_hw3/checklist.json").write_text(json.dumps({k: bool(v) for k,v in check_items.items()}, ensure_ascii=False, indent=2))

```

```text
[통과] 보고서 실행 커널 LeRobot 0.6.x
[통과] 학습 데이터셋 공개 접근 (스냅샷)
[통과] 통제 변수 및 전체 설정 일치
[통과] 실제 GIF 3개 및 대표 클립 평가 명시
[통과] 손실 CSV 15개 체크포인트 값과 산출 방식 명시
[통과] chunk10 공개 모델·가중치 존재
[통과] chunk10 공개 rollout_ 평가 데이터셋 10회
[통과] chunk10 회차별 성공·부드러움·위치 10개 관찰
[통과] chunk10 실패 원인 기록
[통과] chunk50 공개 모델·가중치 존재
[통과] chunk50 공개 rollout_ 평가 데이터셋 10회
[통과] chunk50 회차별 성공·부드러움·위치 10개 관찰
[통과] chunk50 실패 원인 기록
[통과] chunk100 공개 모델·가중치 존재
[통과] chunk100 공개 rollout_ 평가 데이터셋 10회
[통과] chunk100 회차별 성공·부드러움·위치 10개 관찰
[통과] chunk100 실패 원인 기록
[통과] 동일 물체·시작 위치·평가 프로토콜 확인
[통과] 반응성 테스트 9회 관찰
[통과] Temporal ensemble 회차별 결과 10개 입력
[통과] Temporal ensemble 평가 데이터셋 메타데이터 10개
실험 결과 입력 완료. 일부 메타데이터 확인은 미흡으로 남음.

```

```text
756
```

## 부록 B. 보존 자료 및 후속 작업
- 초기 모델: go234/act_chunk10, go234/act_chunk50, go234/act_chunk100 (batch8/AMP=false).
- 초기 원본·출력: archive_hw3/assignment_3_initial_results.ipynb (원본 그대로).
- 업로드 양식: archive_hw3/assignment_3_uploaded_template.ipynb.
- 새 공개 설정: evidence_hw3/retrained_hub_snapshot.json.
- 남은 작업: 기본 평가 30회, 반응성 9회, TE 10회 실행 → 실제 Hub ID·회차별 관찰·새 손실 자료 입력 → 보고서 재실행.
